# Module 5 â€” ML Forecasting with LightGBM
**Type:** [Code With Me]  
**Time:** 45 minutes  
**Job:** Prove that a feature-engineered ML pipeline beats the statistical baselines. Quantify the compute cost. Make an honest recommendation about whether the improvement justifies the infrastructure.

---

> **Instructor note:** This is the most technically dense module. Pace it in four blocks:
> - 5.1â€“5.3: Setup, panel load, feature framing (5 min)
> - 5.4â€“5.6: Configure MLForecast, define features, run CV (15 min, Code With Me)
> - 5.7â€“5.9: Conformal intervals, plot, score (12 min, Code With Me)
> - 5.10â€“5.12: Load full artifact, leaderboard update, enterprise cliff (8 min)
>
> The conformal interval construction in 5.8 is the hardest conceptual step. Budget 3 minutes for it.

---
## 5.1 â€” Setup
**[Watch Only]**

> **Instructor note (1 min):** Run silently. Note the new import: `MLForecast` and `LGBMRegressor`. Everything else is the same pattern as Modules 3 and 4.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

matplotlib.rcParams['figure.figsize'] = (14, 4)
matplotlib.rcParams['axes.spines.top'] = False
matplotlib.rcParams['axes.spines.right'] = False

import lightgbm as lgb
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean

from config import (
    ARTIFACT_DIR,
    PARAMS_DIR,
    HORIZON,
    SEASON_LENGTH,
    N_WINDOWS,
    STEP_SIZE,
    REFIT,
    MICRO_SUBSET_N,
    WORKSHOP_SUBSET_N,
    RANDOM_SEED,
    USE_TUNED_PARAMS,
)
from src.checkpointing import load_checkpoint
from src.evaluation import score_forecasts
from src.schemas import validate_forecast

print("Setup complete.")
print(f"USE_TUNED_PARAMS = {USE_TUNED_PARAMS}")

---
## 5.2 â€” Load Panel and Micro Subset
**[Watch Only]**

> **Instructor note (30 sec):** Same pattern as Module 4. Load from the validated artifact, select top-N by volume.

In [ ]:
panel = load_checkpoint("03_validated_panel")

top_series = (
    panel.groupby("unique_id")["y"]
    .sum()
    .sort_values(ascending=False)
    .head(MICRO_SUBSET_N)
    .index
)
micro = panel[panel["unique_id"].isin(top_series)].copy()

print(f"Micro panel: {micro['unique_id'].nunique()} series, {len(micro):,} rows")

---
## 5.3 â€” Why ML Forecasting?
**[Watch Only]**

> **Instructor note (2 min):** Make the cost argument explicit before writing any code. This section answers the question students always have: "if AutoETS is good, why bother with LightGBM?"

AutoETS is a strong univariate model. It sees one series at a time. It cannot learn patterns that span series â€” shared seasonal profiles, cross-SKU promotional lifts, or price elasticity signals.

**LightGBM via MLForecast is a global model.** It trains on all series simultaneously. Each series becomes a set of training rows. The model learns patterns across the entire category portfolio.

**The compute tradeoff is real:**  
AutoETS on 1,000 series takes seconds. LightGBM on 1,000 series with lag features takes longer and produces a model artifact you now have to version, deploy, and monitor. The accuracy gain must justify that operational cost.

**The features we will build:**

| Feature type | Examples | Business signal |
|---|---|---|
| Autoregressive lags | lag_7, lag_14, lag_28 | Recent demand history |
| Rolling statistics | 28-day rolling mean | Trend smoothing |
| Date features | day_of_week, month | Calendar seasonality |

Adding lags in a notebook is a one-liner. Maintaining a feature store that recomputes these lags daily across 100,000 SKUs â€” with late data, backfills, and audit trails â€” is a data engineering project that takes months.

---
## 5.4 â€” Load LightGBM Parameters
**[Watch Only]**

> **Instructor note (1 min):** Load from `params/` based on the `USE_TUNED_PARAMS` toggle in config. The params files are pre-tested â€” do not change them live. Show students what is in the file.

In [ ]:
params_file = "mlforecast_lgb_tuned.json" if USE_TUNED_PARAMS else "mlforecast_lgb_default.json"
params_path = PARAMS_DIR / params_file

with open(params_path) as f:
    lgb_params = json.load(f)

# Always lock the random seed from config â€” do not let params file override it
lgb_params["random_state"] = RANDOM_SEED

print(f"Loading params from: {params_file}")
for k, v in lgb_params.items():
    print(f"  {k:<22}: {v}")

---
## 5.5 â€” Configure MLForecast
**[Code With Me â€” 3 lines]**

> **Instructor note (4 min):** This is the most important Code With Me cell in the module. Students fill in the `lags`, `lag_transforms`, and `date_features`. Walk through each argument:
> - `lags`: explicit integer list â€” `[7, 14, 28]`. These align to weekly, biweekly, and monthly demand patterns.
> - `lag_transforms`: rolling mean on lag 7, window 28 â€” smooths the weekly lag.
> - `date_features`: `dayofweek` and `month` â€” the two calendar signals that matter most for retail.
>
> Pause and check that everyone has the same config before running.

In [ ]:
# Configure MLForecast â€” all feature engineering happens here, not in a separate step

mlf = MLForecast(
    models=[lgb.LGBMRegressor(**lgb_params)],
    freq="D",
    lags=[7, 14, 28],                               # __FILL_IN__: weekly, biweekly, monthly lags
    lag_transforms={
        7: [RollingMean(window_size=28)],            # __FILL_IN__: rolling mean on lag 7, window 28
    },
    date_features=["dayofweek", "month"],            # __FILL_IN__: day-of-week and month
)

print("MLForecast configured.")
print(f"  Lags          : {mlf.ts.lags}")
print(f"  Date features : {mlf.ts.date_features}")
print(f"  Model         : {next(iter(mlf.models.values())).__class__.__name__}")

**Expected output:**
```
MLForecast configured.
  Lags          : [7, 14, 28]
  Date features : ['dayofweek', 'month']
  Model         : LGBMRegressor
```

---
## 5.6 â€” Run Cross-Validation on the Micro Subset
**[Watch Only]**

> **Instructor note (3 min):** Run and let it execute. LightGBM on 50 series with these features should complete in 15â€“25 seconds. While it runs, explain the recursive forecasting strategy: MLForecast feeds its own predictions back as lag inputs for each step in the horizon. This is why lag features create dependency chains.

In [ ]:
%%time
# Cross-validation on micro subset
# Target runtime: < 30 seconds on Colab CPU

cv_ml_micro = mlf.cross_validation(
    df=micro,
    h=HORIZON,
    n_windows=N_WINDOWS,
    step_size=STEP_SIZE,
    refit=REFIT,
)

print(f"ML CV complete. Shape: {cv_ml_micro.shape}")
print(f"Columns: {list(cv_ml_micro.columns)}")
print(cv_ml_micro.head(3).to_string())

**Expected output:**
```
ML CV complete. Shape: (4200, 5)
Columns: ['ds', 'cutoff', 'y', 'LGBMRegressor']
```

> **Instructor note:** MLForecast returns wide format like StatsForecast. Note there are no interval columns yet â€” we derive them via conformal prediction in 5.7.

---
## 5.7 â€” Reshape and Add Conformal Prediction Intervals
**[Code With Me â€” 2 lines]**

> **Instructor note (4 min):** This is the hardest conceptual step. Spend time on the conformal interval idea:
> - We compute residuals (actual - predicted) across all training windows
> - We take the 10th and 90th percentiles of those residuals
> - We shift the point forecast up and down by those percentiles to form the interval
> - This is empirical, assumption-free, and scales to any model
>
> Students fill in the residual calculation and the percentile extraction. Two lines.

In [ ]:
# Reshape MLForecast wide output to forecast schema
def reshape_mlforecast_cv(cv_df: pd.DataFrame, model_col: str, stage: str) -> pd.DataFrame:
    """
    Reshape MLForecast wide CV output to forecast schema.
    Adds conformal prediction intervals derived from in-sample residuals.
    """
    df = cv_df.reset_index().copy()
    df = df.rename(columns={model_col: "y_hat"})
    df["model"] = "LightGBM"
    df["stage"] = stage
    df["y_hat"] = df["y_hat"].clip(lower=0)

    # Conformal prediction intervals from in-sample residuals
    # Residual = actual - predicted. Negative residual = over-forecast.
    residuals = df["y"] - df["y_hat"]                    # __FILL_IN__: compute residuals

    lo_offset = np.percentile(residuals, 10)             # __FILL_IN__: 10th percentile of residuals
    hi_offset = np.percentile(residuals, 90)

    df["lo_80"] = (df["y_hat"] + lo_offset).clip(lower=0)
    df["hi_80"] = (df["y_hat"] + hi_offset).clip(lower=0)

    return df[["unique_id", "ds", "y", "model", "y_hat", "lo_80", "hi_80", "cutoff", "stage"]]


ml_micro = reshape_mlforecast_cv(cv_ml_micro, model_col="LGBMRegressor", stage="ml")

print(f"Reshaped ML forecasts: {ml_micro.shape}")
print(f"Columns: {list(ml_micro.columns)}")
print(f"Interval sample (lo_80, y_hat, hi_80):")
print(ml_micro[["lo_80", "y_hat", "hi_80"]].describe().round(2).to_string())

**Expected output:**
```
Reshaped ML forecasts: (4200, 9)
Columns: ['unique_id', 'ds', 'y', 'model', 'y_hat', 'lo_80', 'hi_80', 'cutoff', 'stage']
```

---
## 5.8 â€” Plot: LightGBM vs Best Baseline
**[Watch Only]**

> **Instructor note (2 min):** Load the SeasonalNaive micro forecast from the baseline micro artifact for visual comparison. Show both on screen. Ask the room: does LightGBM visually improve over SeasonalNaive? The answer is often "not obviously" â€” which is exactly why we score it.

In [ ]:
# Compare LightGBM vs SeasonalNaive on a single series
sample_uid = top_series[0]
sample_cut = ml_micro["cutoff"].unique()[-1]

actuals   = panel[panel["unique_id"] == sample_uid].set_index("ds")["y"]
plot_start = sample_cut - pd.Timedelta(days=42)

lgbm_fcast = ml_micro[
    (ml_micro["unique_id"] == sample_uid) &
    (ml_micro["cutoff"]    == sample_cut)
].set_index("ds")

# Load micro baselines for SeasonalNaive comparison
try:
    baseline_micro_path = ARTIFACT_DIR / "04_baseline_micro_forecasts.parquet"
    baseline_micro = pd.read_parquet(baseline_micro_path)
    snaive_fcast = baseline_micro[
        (baseline_micro["unique_id"] == sample_uid) &
        (baseline_micro["cutoff"]    == sample_cut) &
        (baseline_micro["model"]     == "SeasonalNaive")
    ].set_index("ds")
    has_baseline = True
except Exception:
    has_baseline = False

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(actuals[actuals.index >= plot_start].index,
        actuals[actuals.index >= plot_start].values,
        color="#333", linewidth=1.0, label="Actual", zorder=3)

ax.plot(lgbm_fcast.index, lgbm_fcast["y_hat"],
        color="#7B1FA2", linewidth=1.5, linestyle="--", label="LightGBM", zorder=4)
ax.fill_between(lgbm_fcast.index, lgbm_fcast["lo_80"], lgbm_fcast["hi_80"],
                alpha=0.15, color="#7B1FA2", label="LightGBM 80% interval")

if has_baseline and len(snaive_fcast) > 0:
    ax.plot(snaive_fcast.index, snaive_fcast["y_hat"],
            color="#1E88E5", linewidth=1.2, linestyle=":", label="SeasonalNaive", zorder=3)

ax.axvline(sample_cut, color="#999", linestyle=":", linewidth=1)
ax.set_title(f"LightGBM vs SeasonalNaive â€” {sample_uid} (Window 3)", fontsize=11)
ax.set_ylabel("Units sold")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

**Expected output:** LightGBM forecast with shaded 80% interval alongside SeasonalNaive. The interval width communicates model confidence â€” compare it to the SeasonalNaive line visually.

---
## 5.9 â€” Score the Micro ML Forecasts
**[Code With Me â€” 1 line]**

> **Instructor note (2 min):** Students validate and score the micro results. One fill-in: the `validate_forecast` call. The point is to make schema validation a reflex â€” run it every time before scoring.

In [ ]:
# Validate schema before scoring
ml_micro_validated = validate_forecast(ml_micro, artifact_name="05_ml_micro")  # __FILL_IN__: validate ml_micro

ml_micro_scores = score_forecasts(
    ml_micro_validated,
    subset_name=f"micro_{MICRO_SUBSET_N}",
)

wmape_row  = ml_micro_scores[ml_micro_scores["metric"] == "wMAPE"].iloc[0]
iscore_row = ml_micro_scores[ml_micro_scores["metric"] == "IntervalScore_80"].iloc[0]
bias_row   = ml_micro_scores[ml_micro_scores["metric"] == "Bias"].iloc[0]

print(f"LightGBM on micro subset ({MICRO_SUBSET_N} series):")
print(f"  wMAPE          : {wmape_row['score']:.4f}")
print(f"  Interval Score : {iscore_row['score']:.4f}")
print(f"  Bias           : {bias_row['score']:+.4f}  (+ = over-forecast, - = under-forecast)")

**Expected output:**
```
LightGBM on micro subset (50 series):
  wMAPE          : 0.XXXX
  Interval Score : X.XXXX
```

> **Instructor note:** Do not read the micro score as final. The micro subset is the top-50 by volume â€” the easiest 50 to forecast. The full-subset score will be higher (worse). This is a known selection bias.

---
## 5.10 â€” Load Full-Subset ML Forecasts and Update Leaderboard
**[Watch Only]**

> **Instructor note (2 min):** Load the precomputed full-subset ML artifact, score it, and compare against the baseline floor. This is the first real leaderboard update â€” does LightGBM beat AutoETS?

In [ ]:
# ðŸ”´ RED PATH (standard) â€” load precomputed full-subset ML forecasts
ml_full = load_checkpoint("05_ml_forecasts")

ml_full_scores = score_forecasts(
    ml_full,
    subset_name=f"workshop_{WORKSHOP_SUBSET_N}",
)

# Load baseline scores for comparison
baseline_full = load_checkpoint("04_baseline_forecasts")
baseline_scores = score_forecasts(
    baseline_full,
    subset_name=f"workshop_{WORKSHOP_SUBSET_N}",
)

# Combine into updated leaderboard
from src.evaluation import build_leaderboard
leaderboard_5 = build_leaderboard([baseline_scores, ml_full_scores])

print("Updated leaderboard (wMAPE):")
wmape_lb = leaderboard_5[["model", "wMAPE", "Bias"]].dropna().sort_values("wMAPE") if "wMAPE" in leaderboard_5.columns else leaderboard_5
print(wmape_lb.to_string(index=False))

**Expected output:**
```
Updated leaderboard (wMAPE):
          model    wMAPE
       LightGBM   0.XXXX
        AutoETS   0.XXXX
  SeasonalNaive   0.XXXX
Chronos-t5-mini   0.XXXX
          Naive   0.XXXX
```

> **Instructor note:** LightGBM should beat AutoETS on wMAPE for most M5 subsets. If it does not, the feature pipeline needs review. The gap between them is the quantified value of the ML pipeline.

---
## 5.11 â€” Save the Micro ML Artifact
**[Watch Only]**

> **Instructor note (30 sec):** Save the micro results for optional take-home analysis. The full-subset artifact is already loaded.

In [ ]:
micro_ml_path = ARTIFACT_DIR / "05_ml_micro_forecasts.parquet"
ml_micro_validated.to_parquet(micro_ml_path, index=False)
print(f"  âœ“ Micro ML artifact saved : {micro_ml_path.name} ({len(ml_micro_validated):,} rows)")
print(f"  âœ“ Full ML artifact loaded : 05_ml_forecasts.parquet ({len(ml_full):,} rows)")

---
## 5.12 â€” The Enterprise Cliff
**[Watch Only]**

> **Instructor note (2 min):** This is the authority hook. Say the first paragraph slowly â€” it is the line that differentiates practitioners from notebook runners.

Adding lags in a notebook is a one-liner. **Orchestrating a scalable feature pipeline across 100,000 SKUs is where enterprise forecasting systems diverge from standalone scripts.**

What we simplified:

**Lag computation at scale:**  
MLForecast computes lags on the fly during CV. In production, lags are computed once, stored in a feature store, versioned, and served to the model at inference time. Late-arriving data invalidates precomputed lags â€” your pipeline needs a backfill strategy.

**Conformal intervals are a simplification:**  
The residual-based intervals we computed assume that in-sample errors are representative of out-of-sample errors. That holds on stable series. On series with trend shifts or promotional events, it breaks. Production-grade conformal calibration requires held-out calibration sets, not in-sample residuals.

**The ROI question:**  
LightGBM beat AutoETS by some margin on wMAPE. The question your CFO asks is: how many basis points of safety stock reduction does that margin translate to? If the answer is less than the infrastructure cost of maintaining a feature store, AutoETS is the right production choice.

---

> **Instructor note â€” transition:** "We have a working ML pipeline and a leaderboard with 5 models. One more to add â€” a global neural model."
>
> Open `instructor_notebooks/06_deep_learning.ipynb`.